## TUGAS_MANDIRI

**Nama : Cahyo Adi Nugroho**

**NPM : 2505060034**

**Rombel Praktikum : 2**

In [1]:
#memulai sparksession di notebook baru

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("TugasMandiri4") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

26/09/13 17:47:01 WARN Utils: Your hostname, cahyoalim-ThinkPad-L14-Gen-2 resolves to a loopback address: 127.0.1.1; using 192.168.1.28 instead (on interface wlp9s0)
26/09/13 17:47:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/13 17:47:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
#A. membaca dan eksplorasi data

#menentukan lokasi berjas data di dalaam hdfs
path_hdfs = "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv"
df = spark.read.csv(path_hdfs, header=True, inferSchema=True)

#menampilkan skema
df.printSchema()

#menampilkan 10 baris pertama
df.show(10)

#menampilkan total baris
print("Total baris : ", df.count())


root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000| 

In [3]:
#B. Menangani Data Kosong 
from pyspark.sql.functions import col

#menghitung jumlah baris yang memiliki nilai kosong pada kolom rating
jumlah_null = df.filter(col("rating").isNull()).count()
print(f"jumlah baris kosong pada kolom (NULL) pada kolom rating: {jumlah_null}")

#memilih na.fill() untuk penanganan nilai null pada kolom rating
penanganan = df.na.fill({"rating":0})

#memastikan bahwa nilai null sudah tidak ada lagi
jumlah_null2 = penanganan.filter(col("rating").isNull()).count()
print(f"jumlah null pada kolom rating setelah ditangani: {jumlah_null2}")

jumlah baris kosong pada kolom (NULL) pada kolom rating: 204
jumlah null pada kolom rating setelah ditangani: 0


**ALASAN SAYA MEMILIH na.fll()**


na.fill() akan menggantikan nilai null menjadi 0 sedangkan na.drop() akan menghapus baris yang memiliki nilai null pada kolom rating. Jika saya memilih na.drop() maka seluruh data yang memiliki rating null akan dihapus dan ini akan menyebabkan analisis proses bisnis menjadi terganggu karena jumlah data yang memiliki nilai null pada kolom rating sebanyak 204 atau 20% lebih.

In [4]:
#C. Transfromasi data
from pyspark.sql.functions import col, when

#menambahkan kolom total_pendapatan
transformasi = penanganan.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))

#menambahkakn kolom tier_transaksi 
transformasi = transformasi.withColumn("tier_transaksi",when(col("total_pendapatan") > 500000, "besar").otherwise("kecil"))

#menampilkan 10 baris pertam
transformasi.show(10)

+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|          270000|         kecil|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|          600000|         besar|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|          480000|         kecil|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|         21

In [5]:
#D. Analisis dengan GroupBy
from pyspark.sql.functions import sum as spark_sum, count, avg, col

#kategori dengan total pendapatan paling tinggi
print("1. Kategori dengan total pendapatan tertinggi")
transformasi.groupBy("kategori").agg(spark_sum("total_pendapatan").alias("total_pendapatan")).orderBy(col("total_pendapatan").desc()).show(1)

#kota dengan jumlah tier transaksi 'besar' terbanyak
print("2. Kota dengan tier tansaksi 'besar' terbanyak")
transformasi.filter(col("tier_transaksi") == "Besar").groupBy("kota").agg(count("order_id").alias("transaksi_besar")).orderBy(col("transaksi_besar").desc()).show(1)

#rata_rata rating untuk masing-masing metode pemayaran
print("3. rata-rata ratig per metode pembayaran")
transformasi.groupBy("metode_pembayaran").agg(avg("rating").alias("rata_rata_rating")).show()


1. Kategori dengan total pendapatan tertinggi
+------------+----------------+
|    kategori|total_pendapatan|
+------------+----------------+
|Rumah Tangga|       138665000|
+------------+----------------+
only showing top 1 row

2. Kota dengan tier tansaksi 'besar' terbanyak
+----+---------------+
|kota|transaksi_besar|
+----+---------------+
+----+---------------+

3. rata-rata ratig per metode pembayaran
+-----------------+------------------+
|metode_pembayaran|  rata_rata_rating|
+-----------------+------------------+
|              COD|3.3745019920318726|
|    Transfer Bank|3.3399209486166006|
|     Kartu Kredit|3.1910569105691056|
|         E-Wallet|             3.292|
+-----------------+------------------+



In [6]:
#E. Menyimpan Hasil ke HDFS

#Menentukan path tujuan di HDFS
path_simpan_hdfs = "hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_transformasi_csv"

#Menyimpan DataFrame hasil transformasi C ke HDFS dalam format CSV
transformasi.write.mode("overwrite").csv(path_simpan_hdfs, header=True)
print("Berhasil menyimpan data ke HDFS!")

# Verifikasi: Membaca kembali data yang baru disimpan dari HDFS
df_verifikasi = spark.read.csv(path_simpan_hdfs, header=True, inferSchema=True)
print("Verifikasi Berhasil! Jumlah baris data yang dibaca kembali:", df_verifikasi.count())
df_verifikasi.show(5)

Berhasil menyimpan data ke HDFS!
Verifikasi Berhasil! Jumlah baris data yang dibaca kembali: 1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|          270000|         kecil|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|          600000|         besar|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|          480000|         kecil|
|ORD-3003|2026-09-09 00:00

**Mengapa Hasil Simpanan Spark Terdiri dari Beberapa Berkas Partisi (part-00000...)?**

jawaban : Spark menyimpan hasil olahan data ke dalam bentuk beberapa berkas partisi (part-00000-....csv, part-00001-....csv, dst.) dan bukan satu berkas tunggal karena Spark bekerja menggunakan arsitektur komputasi terdistribusi atau paralel.  Data dibagi ke dalam beberapa blok yang diproses secara independen oleh Core CPU/Executor yang berbeda. Ketika proses penyimpanan dilakukan, setiap Executor menuliskan bagian datanya masing-masing secara bersamaan (paralel) langsung ke HDFS untuk mengoptimalkan kecepatan write (throughput).  Hal ini berbeda dengan pustaka single-threaded seperti pandas yang menulis seluruh data ke dalam satu berkas tunggal secara sekuensial. 

In [7]:
spark.stop()